# Experiment 9 - Part 2: A Stateful Agent in LangGraph

**Course:** HAAI302COM - Artificial Intelligence Laboratory

The hand-written loop of notebook 01 works, but it has no memory between questions, no way to pause, and no structure we can inspect. **LangGraph** makes the same loop a *graph*:

- **State**: a shared record that every step reads and updates (here: the messages, a step counter and a list of tools used).
- **Nodes**: Python functions that take the state and return a *partial update* to it.
- **Edges**: who runs next. A **conditional edge** chooses the next node by looking at the state.
- **Checkpointer**: saves the state after every step, which gives us memory across turns, replay, and (next notebook) pausing for a human.

We build the agent as a graph, watch the state change step by step, add a step limit, add memory, and finally measure the agent against a model without tools.

## 0. Setup

In [1]:
import json, operator, time, warnings
warnings.filterwarnings("ignore")
from typing import Annotated, TypedDict
import pandas as pd
import agent_tools as at
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 170)
at.enable_cache()
llm_tools = at.get_llm().bind_tools(at.TOOLS)
SYSTEM = SystemMessage("You are a careful assistant. Use the calculator for arithmetic, run_python for code and date "
                       "calculations, and web_search for facts you are not sure about. Do not use a tool for simple "
                       "conversation or well-known facts. Answer concisely. After a tool returns, always reply to the user with the "
                       "final answer in one sentence.")
TERSE = SystemMessage(SYSTEM.content.replace(" After a tool returns, always reply to the user with the final answer in one sentence.", ""))

## 1. The smallest agent graph

`MessagesState` is a ready-made state with one field, `messages`, whose update rule is *append* (so a node returns only the new messages). We add two nodes:

- `agent`: calls the model. The model's reply may contain tool calls.
- `tools`: LangGraph's `ToolNode` runs every tool call in the last message and returns `ToolMessage`s.

Edges: `START -> agent`. After `agent`, a **conditional edge** looks at the last message: tool calls present means go to `tools`, otherwise `END`. After `tools` we always go back to `agent`. That cycle is the agent loop.

In [2]:
from langgraph.graph import MessagesState
from langgraph.prebuilt import tools_condition      # routes to "tools" if the last message has tool calls, else END

def agent(state: MessagesState):
    return {"messages": [llm_tools.invoke([SYSTEM] + state["messages"])]}

g = StateGraph(MessagesState)
g.add_node("agent", agent)
g.add_node("tools", ToolNode(at.TOOLS))
g.add_edge(START, "agent")
g.add_conditional_edges("agent", tools_condition)
g.add_edge("tools", "agent")
app = g.compile()
print(app.get_graph().draw_ascii())

        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +-------+           
          | agent |           
          +-------+.          
          .         .         
        ..           ..       
       .               .      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


**What to notice:** the drawing shows the whole agent: START goes to agent, the dotted lines out of agent are the conditional edge (to tools, or to END), and tools goes back to agent. That cycle is the loop from notebook 01. The code is about six lines: two nodes, one normal edge in, one conditional edge, one edge back. Nodes return only the partial update (here, the new message), and LangGraph merges it into the state.

## 2. Watching the state change

`app.invoke` returns only the final state. `app.stream(..., stream_mode="updates")` yields one item per node that ran: `{node_name: the_update_it_returned}`. This is the best debugging tool for a graph. We run a question that needs two tools and print one row per step.

In [3]:
q = "Find the year Python was first released using a web search, then compute how many years before 2025 that was."
rows = []
for i, update in enumerate(app.stream({"messages": [HumanMessage(q)]}, stream_mode="updates"), 1):
    for node, upd in update.items():
        m = upd["messages"][-1]
        what = f"tool_calls={[(c['name'], c['args']) for c in m.tool_calls]}" if getattr(m, "tool_calls", None) else m.content[:90]
        rows.append(dict(step=i, node=node, message_type=type(m).__name__, content=what))
pd.DataFrame(rows)

,step,node,message_type,content
0,1,agent,AIMessage,"tool_calls=[('web_search', {'query': 'when was python first released'})]"
1,2,tools,ToolMessage,- History of Python - Wikipedia: The programming language Python was conceived in the ...
2,3,agent,AIMessage,"tool_calls=[('calculator', {'expression': '2025 - 1991'})]"
3,4,tools,ToolMessage,34
4,5,agent,AIMessage,"Python was first released in 1991, which was 34 years before 2025."


**What to notice:** five updates alternate between agent and tools: the agent asks for web_search with the query 'when was python first released', the tools node returns the search text, the agent asks for a calculation (2025 - 1991) and gets 34, and the final agent step writes the answer. Compared with notebook 01 the model chose the calculator here where it chose run_python before: with the same question the tool choice can vary between runs and prompts, a reason to log every step.

## 3. Custom state: a counter, a log and a step limit

An agent that never stops is a bill and a risk. We extend the state with two fields:

- `steps`: how many times the model has been called. A plain field: each update **replaces** the value.
- `tools_used`: a list. We declare it with `Annotated[list, operator.add]`; this *reducer* tells LangGraph to **concatenate** updates instead of replacing them.

A new router decides after each model call: no tool calls means finish; too many steps means go to a `give_up` node that writes a polite failure message; otherwise run the tools.

In [4]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]       # reducer: append messages
    steps: int                                    # no reducer: replaced on every update
    tools_used: Annotated[list, operator.add]     # reducer: concatenate lists

def make_agent(system):
    def agent2(state: AgentState):
        ai = llm_tools.invoke([system] + state["messages"])
        return {"messages": [ai], "steps": state.get("steps", 0) + 1, "tools_used": [c["name"] for c in ai.tool_calls]}
    return agent2

def give_up(state: AgentState):
    return {"messages": [AIMessage("I could not finish within the step limit.")]}

In [5]:
def nudge(state: AgentState):
    return {"messages": [HumanMessage("You did not write an answer. State the final answer in one short sentence, using the tool results above.")]}

def build_graph(max_steps=5, checkpointer=None, retry_empty=False, system=SYSTEM):
    def route(state: AgentState):
        last = state["messages"][-1]
        if not last.tool_calls:
            if retry_empty and not last.content.strip() and state["steps"] < max_steps:
                return "nudge"                                   # the model returned an empty answer: ask once more
            return END
        return "give_up" if state["steps"] >= max_steps else "tools"
    g = StateGraph(AgentState)
    g.add_node("agent", make_agent(system)); g.add_node("tools", ToolNode(at.TOOLS)); g.add_node("give_up", give_up); g.add_node("nudge", nudge)
    g.add_edge(START, "agent")
    g.add_conditional_edges("agent", route, ["tools", "give_up", "nudge", END])
    g.add_edge("tools", "agent"); g.add_edge("give_up", END); g.add_edge("nudge", "agent")
    return g.compile(checkpointer=checkpointer)

app2 = build_graph(max_steps=5)
out = app2.invoke({"messages": [HumanMessage(q)], "steps": 0, "tools_used": []})
print("steps:", out["steps"], "| tools used:", out["tools_used"], "| messages:", len(out["messages"]))
print("answer:", out["messages"][-1].content)

steps: 3 | tools used: ['web_search', 'calculator'] | messages: 6
answer: Python was first released in 1991, which was 34 years before 2025.


**What to notice:** after the run, steps is 3 (three model calls), tools_used is ['web_search', 'calculator'] and there are 6 messages. The two behaviours differ by their reducers: tools_used grew by concatenation across the nodes' updates (operator.add), while steps was replaced each time by the new count.

Now with the limit set too low for the same two-step question: `max_steps=1` lets the model call its first tool but forbids a second round.

In [6]:
out = build_graph(max_steps=1).invoke({"messages": [HumanMessage(q)], "steps": 0, "tools_used": []})
print("steps:", out["steps"], "| tools used:", out["tools_used"])
print("answer:", out["messages"][-1].content)

steps: 1 | tools used: ['web_search']
answer: I could not finish within the step limit.


**What to notice:** with max_steps=1 the model made its first call (a web search), the router saw a tool request while steps had already reached the limit, and sent the run to give_up. The user receives a clear message instead of a runaway loop. The cost: the search had already been paid for and its result is discarded. A kinder policy could return the partial results.

## 4. A second safety net: the recursion limit

LangGraph also counts every node execution, and raises `GraphRecursionError` when a run exceeds `recursion_limit` (default 25). It protects against bugs in the *graph* (a cycle with no exit), while our `max_steps` is a *policy* about the model. We show the error by giving a run only 3 super-steps.

In [7]:
from langgraph.errors import GraphRecursionError
try:
    app.invoke({"messages": [HumanMessage(q)]}, {"recursion_limit": 3})
except GraphRecursionError as e:
    print("GraphRecursionError:", str(e)[:120])

GraphRecursionError: Recursion limit of 3 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limi


**What to notice:** with recursion_limit=3 the run stops after three super-steps (agent, tools, agent) with GraphRecursionError, before the answer is written, and the message says how to raise the limit. The two limits have different jobs: max_steps is our policy about how long the model may work, and the recursion limit is the engine's last-resort protection against a cycle that never ends.

## 5. Memory with a checkpointer

A compiled graph is stateless between calls. Add a **checkpointer** and the state is saved after every step under a `thread_id`. A later call with the same thread id continues from the saved state, so the model sees the earlier messages. `MemorySaver` keeps checkpoints in RAM. We use a fresh thread for each conversation.

In [8]:
mem_app = build_graph(max_steps=5, checkpointer=MemorySaver())
cfg_a = {"configurable": {"thread_id": "ravi"}}
mem_app.invoke({"messages": [HumanMessage("My name is Ravi. What is 12 times 12?")], "steps": 0, "tools_used": []}, cfg_a)
r1 = mem_app.invoke({"messages": [HumanMessage("What is my name, and what number did I just ask you to compute?")], "steps": 0, "tools_used": []}, cfg_a)
r2 = mem_app.invoke({"messages": [HumanMessage("What is my name?")], "steps": 0, "tools_used": []}, {"configurable": {"thread_id": "someone-else"}})
print("same thread     :", r1["messages"][-1].content)
print("different thread:", r2["messages"][-1].content)

same thread     : Your name is Ravi, and you asked me to compute 144.
different thread: I do not have access to personal information, so I do not know your name.



**What to notice:** the first thread remembers: 'Your name is Ravi, and you asked me to compute 144.' A different thread_id knows nothing and says so. The checkpointer saves the state under the thread id, so each user or conversation has its own memory and they never mix.

## 6. Looking inside: state snapshots and history

`get_state(config)` returns the latest **snapshot**: the values, the node that would run next (`next`) and the checkpoint id. `get_state_history(config)` returns all snapshots, newest first. Each row is a saved moment we could return to.

In [9]:
snap = mem_app.get_state(cfg_a)
print("next node:", snap.next, "| messages in state:", len(snap.values["messages"]), "| steps:", snap.values["steps"])
hist = list(mem_app.get_state_history(cfg_a))
pd.DataFrame([dict(checkpoint=h.config["configurable"]["checkpoint_id"][:8], next=h.next, n_messages=len(h.values.get("messages", [])),
                   steps=h.values.get("steps"), source=h.metadata.get("source"), step=h.metadata.get("step")) for h in hist]).head(8)

next node: () | messages in state: 6 | steps: 1


,checkpoint,next,n_messages,steps,source,step
0,1f1d4262,(),6,1.0,loop,6
1,1f1d4262,"(agent,)",5,0.0,loop,5
2,1f1d4262,"(__start__,)",4,2.0,input,4
3,1f1d4262,(),4,2.0,loop,3
4,1f1d4262,"(agent,)",3,1.0,loop,2
5,1f1d4262,"(tools,)",2,1.0,loop,1
6,1f1d4261,"(agent,)",1,0.0,loop,0
7,1f1d4261,"(__start__,)",0,NaN,input,-1


**What to notice:** the finished conversation has 6 messages and nothing left to run (next is empty). The history lists eight checkpoints, newest first, from the empty start to the final answer; each row records which node would run next, such as (tools,) or (agent,). Every one of them could be resumed, which is how replay and time travel work. The steps column goes back to 1 or 2 for the second question because we passed steps 0 with each new input.

## 7. Memory that survives a restart: SQLite

`MemorySaver` disappears when the Python process ends. `SqliteSaver` stores the same checkpoints in a database file. To prove persistence we build a **brand new** graph and a new saver on the same file (like restarting a server) and continue the conversation.

In [10]:
import sqlite3, os
from langgraph.checkpoint.sqlite import SqliteSaver
if os.path.exists("agent_memory.sqlite"):
    os.remove("agent_memory.sqlite")
cfg_s = {"configurable": {"thread_id": "session-1"}}

conn = sqlite3.connect("agent_memory.sqlite", check_same_thread=False)
a1 = build_graph(checkpointer=SqliteSaver(conn))
a1.invoke({"messages": [HumanMessage("Remember this code word: pelican. What is 7 times 6?")], "steps": 0, "tools_used": []}, cfg_s)
conn.close()                                   # the 'server' stops here

conn2 = sqlite3.connect("agent_memory.sqlite", check_same_thread=False)    # a new process would do exactly this
a2 = build_graph(checkpointer=SqliteSaver(conn2))
out = a2.invoke({"messages": [HumanMessage("What was the code word?")], "steps": 0, "tools_used": []}, cfg_s)
print(out["messages"][-1].content, "| database size:", os.path.getsize("agent_memory.sqlite"), "bytes")

The code word was pelican. | database size: 32768 bytes


**What to notice:** after the 'restart' the new graph recalled the code word ('pelican') from the database file, which is about 32 KB. Nothing was kept in Python memory between the two graphs. A web service would do the same on each request: build the graph, give it the thread id, and the saver loads the right conversation.

## 8. Does the agent actually help? An evaluation

We compare two systems on 12 tasks with known answers: the **agent** (the graph above, with tools) and a **plain model** with the same system prompt but no tools. A task is correct if the reply contains the expected text (commas in numbers are ignored). We also record how many model calls and tool calls the agent needed. We run the agent with **three prompt/graph variants** because we observed a failure in this model: with a prompt that only says "answer concisely" it sometimes returns an **empty** final message after a tool result. `terse` is that prompt; `terse_retry` adds a `nudge` node that asks again when the answer is empty; `fixed_prompt` instead tells the model what to do after a tool returns. The last task is a greeting, where the correct behaviour is to use **no** tool.

In [11]:
TASKS = [("What is 48271 multiplied by 9371?", ["452347541"]),
         ("What is the square root of 2 to six decimal places?", ["1.414214"]),
         ("What is 15 percent of 2480 plus 7 squared?", ["421"]),
         ("How many days are there between 2024-03-15 and 2025-11-02?", ["597"]),
         ("What is the sum of the first 50 prime numbers?", ["5117"]),
         ("Reverse the string 'agentic' and count its vowels.", ["citnega", "3"]),
         ("Who created the Python programming language? Search the web.", ["guido"]),
         ("Which company developed the LangGraph library?", ["langchain"]),
         ("What is the factorial of 20?", ["2432902008176640000"]),
         ("Find the year Python was first released using a web search, then compute how many years before 2025 that was.", ["34"]),
         ("What is the capital of France?", ["paris"]),
         ("Hello! Who are you?", [])]
plain = at.get_llm()

def ok(text, keys):
    t = text.lower().replace(",", "")
    return bool(text.strip()) and all(k in t for k in keys)

rows = []
terse_app = build_graph(max_steps=6, system=TERSE)                       # prompt that says only 'answer concisely'
retry_app = build_graph(max_steps=6, system=TERSE, retry_empty=True)     # same prompt plus the nudge node
fixed_app = build_graph(max_steps=6)                                     # prompt that also says what to do after a tool returns
for q_, keys in TASKS:
    st = {"messages": [HumanMessage(q_)], "steps": 0, "tools_used": []}
    o1, o2, o3 = terse_app.invoke(st), retry_app.invoke(st), fixed_app.invoke(st)
    base = plain.invoke([SYSTEM, HumanMessage(q_)]).content
    rows.append(dict(task=q_[:48], plain=ok(base, keys), terse=ok(o1["messages"][-1].content, keys),
                     terse_retry=ok(o2["messages"][-1].content, keys), fixed_prompt=ok(o3["messages"][-1].content, keys),
                     tools=",".join(o3["tools_used"]) or "-", answer=repr(o3["messages"][-1].content[:36])))
ev = pd.DataFrame(rows)
print(ev[["plain", "terse", "terse_retry", "fixed_prompt"]].mean().round(2).to_dict())
ev

{'plain': 0.58, 'terse': 0.92, 'terse_retry': 1.0, 'fixed_prompt': 1.0}


,task,plain,terse,terse_retry,fixed_prompt,tools,answer
0,What is 48271 multiplied by 9371?,False,True,True,True,calculator,"'48271 multiplied by 9371 is 452,347,'"
1,What is the square root of 2 to six decimal plac,True,True,True,True,calculator,'The square root of 2 to six decimal '
2,What is 15 percent of 2480 plus 7 squared?,True,True,True,True,calculator,'15 percent of 2480 plus 7 squared is'
3,How many days are there between 2024-03-15 and 2,False,True,True,True,run_python,'There are 597 days between 2024-03-1'
4,What is the sum of the first 50 prime numbers?,False,False,True,True,run_python,'The sum of the first 50 prime number'
5,Reverse the string 'agentic' and count its vowel,False,True,True,True,run_python,"""The reversed string is 'citnega' and"""
6,Who created the Python programming language? Sea,True,True,True,True,web_search,'The Python programming language was '
7,Which company developed the LangGraph library?,True,True,True,True,web_search,'The LangGraph library was developed '
8,What is the factorial of 20?,True,True,True,True,calculator,"'The factorial of 20 is 2,432,902,008'"
9,Find the year Python was first released using a,False,True,True,True,"web_search,calculator","'Python was first released in 1991, w'"


**What to notice:** the plain model scores 0.58 (7 of 12): it fails the large multiplication, the date count, the sum of primes, the string task and the two-step question, and passes the easy and well-known ones. The agent with the terse prompt scores 0.92. Its one miss is the primes task: the tool returned 5117 correctly, then the model ended with an empty message. Two different fixes both reach 1.00: a graph-level retry that asks again when the answer is empty, and one sentence in the prompt saying what to do after a tool returns. So prompt wording is part of an agent's reliability, and a guard in the graph is cheap insurance. Remaining weakness: the agent still searches for 'capital of France' although the prompt says not to use tools for well-known facts.

## 9. Recap and exercises

- A LangGraph agent is state + nodes + edges; the conditional edge is the decision point.
- `stream(..., stream_mode="updates")` shows the work step by step; reducers (`add_messages`, `operator.add`) control how updates merge.
- Guard rails: `max_steps` (policy) and `recursion_limit` (safety).
- A checkpointer gives memory per `thread_id`, history to inspect, and persistence with SQLite.
- Tools help most on exact computation and on facts outside the model's memory, and do nothing for simple conversation.

**Try yourself:** (1) add a field `cost` to the state that accumulates tokens used; (2) use `get_state_history` to find the checkpoint before the first tool call and re-run from there with `app.invoke(None, checkpoint_config)`; (3) add a fourth tool (for example the current date).

Next: **Notebook 03** adds human-in-the-loop checkpoints so that risky tool calls wait for approval.